In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import sksurv
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

In [ ]:
df = pd.read_csv('../Resources/Overall_cleaned.csv',encoding='latin-1')
df = df[:400000]
y_time = df['Survival months']
y_event = df['Vital status recode (study cutoff used)']
X = df.drop(columns=['Survival months',
                     'Vital status recode (study cutoff used)'])
#create survival target (used for train test split only)
y_struct = Surv.from_arrays(event=y_event,time=y_time)
X_train,X_test,y_train,y_test = train_test_split(X,y_struct,test_size=0.2,random_state=42)

In [ ]:
#Only if NO pd.getdummies()
#convert string columns to numeric (ie Female - 0)
from sklearn.preprocessing import LabelEncoder

#object implies text
categorical_cols = X.select_dtypes(include=['object']).columns

label_encoders =  {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

    #save encoder to see mapping
    label_encoders[col] = le

In [ ]:
print(X['SEER Combined Mets at DX-bone (2010+)'].value_counts())

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y_struct,test_size=0.2,random_state=42)

In [ ]:
print(len(X_train))

In [ ]:
#Implement and Fit RSF
#free memory
#Takes time 4min
import gc
gc.collect()

rsf = RandomSurvivalForest(n_estimators=30, min_samples_leaf=10, min_samples_split=5, n_jobs=3, low_memory=True,verbose=1,random_state=42)
rsf.fit(X_train, y_train)

In [ ]:
gc.collect()
rsf.set_params(n_jobs=3)
print("Test c-index:", rsf.score(X_test, y_test))
#not hot encoding leads to worse c index

In [ ]:
import skopt.space.space as s

search_space = list()

search_space.append(s.Integer(10,100,name='n_estimators'))
search_space.append(s.Integer(2,50,name='min_samples_split'))
search_space.append(s.Integer(5,30,name='min_samples_leaf'))
search_space.append(s.Integer(5,35,name='max_features'))
search_space.append(s.Integer(50,300,name='max_leaf_nodes'))
search_space.append(s.Integer(10000,int(len(X_train)*0.5),name='max_samples'))

In [ ]:
#gibes precision score
from sklearn.model_selection import cross_val_score
#for cross validation
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from numpy import mean
from numpy import std
import numpy as np
import sys
import importlib
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

def evaluate_model(params,train_idx,test_idx):
    num_curves = 10
    n_estimators,min_samples_split,min_samples_leaf, max_features, max_leaf_nodes, max_samples = params
    model = RandomSurvivalForest(n_estimators=n_estimators,min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,max_features=max_features,max_leaf_nodes=max_leaf_nodes,max_samples=max_samples,n_jobs=5)
    #define the splits
    
    count=0
    
    #apply the splits 
    
    count+=1 #check that it's 30 iterations
    print(f"count is {count}")
    subset_X_train = X_train.iloc[train_idx]
    subset_y_train = y_train[train_idx]
    subset_X_test = X_train.iloc[test_idx]
    subset_y_test = y_train[test_idx]

    model.fit(subset_X_train,subset_y_train)
    model_scores = model.predict(subset_X_train)
    #create num_curves equal width bins based on training range
    min_score, max_score = model_scores.min(), model_scores.max()
    bins = np.linspace(min_score, max_score, num_curves+1) #num_curves+1 edges = num_curves bins

    #assign the test samples to corresponding bins, thus assigning labels
    subset_test_scores = model.predict(subset_X_test)
    subset_test_risk_groups = pd.cut(subset_test_scores, bins=bins, labels=range(1, num_curves+1), include_lowest=True)
    subset_X_test_extended = subset_X_test.copy()
    subset_X_test_extended = pd.DataFrame(subset_X_test,columns=X_test.columns)
    subset_X_test_extended['Risk_Label'] = subset_test_risk_groups.map(lambda x: f'EL{x}')
    subset_X_test_extended['Survival_Time'] = subset_y_test['time']
    subset_X_test_extended['Event_Status'] = subset_y_test['event']

    km_plotter = KM_Plotter(subset_X_test_extended)
    km_plotter.makeKMPlot(plot=False)
    score = km_plotter.getKMScore()
    print(f"Score for fold is: {score}")
    return score
    


In [ ]:
import skopt
from sklearn.model_selection import RepeatedStratifiedKFold
cv = RepeatedStratifiedKFold(n_splits=2,n_repeats=1,random_state=1)
best_scores = []
best_configs = []
for train_idx, test_idx in cv.split(X_train,y_train):
    result = skopt.gp_minimize(
        lambda params: evaluate_model(params, train_idx, test_idx),
        search_space,
        n_jobs=2,
        n_calls=50,
        random_state=42
    )
    best_scores.append(result.fun)
    best_configs.append(result.x)
    

In [ ]:
import skopt
result = skopt.gp_minimize(evaluate_model,search_space,n_jobs=3, n_calls = 10)

In [ ]:
print(best_configs)
print(best_scores)

In [ ]:
#demo testing - want to understand variables in KMPlotter
num_curves = 10
n_estimators,min_samples_split,min_samples_leaf, max_features, max_leaf_nodes, max_samples = best_configs[1]
model = RandomSurvivalForest(n_estimators=n_estimators,min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,max_features=max_features,max_leaf_nodes=max_leaf_nodes,max_samples=max_samples,n_jobs=4)
model.fit(X_train, y_train)
train_scores = model.predict(X_train)

min_score, max_score = train_scores.min(), train_scores.max()
bins = np.linspace(min_score, max_score, num_curves+1) #num_curves+1 edges = num_curves bins

test_scores = model.predict(X_test)
test_risk_groups = pd.cut(test_scores, bins=bins, labels=range(1, num_curves+1), include_lowest=True)

X_test_extended = X_test.copy()
X_test_extended = pd.DataFrame(X_test,columns=X_test.columns)
X_test_extended['Risk_Label'] = test_risk_groups.map(lambda x: f'EL{x}')
X_test_extended['Survival_Time'] = y_test['time']
X_test_extended['Event_Status'] = y_test['event']

km_plotter = KM_Plotter(X_test_extended)
km_plotter.makeKMPlot(plot=True)
score = km_plotter.getKMScore()
print(score)

In [ ]:
temp = km_plotter.survival_functions_temp
print(temp)

In [ ]:
#create timeline from 0 to 150 with step size 1
timeline = pd.DataFrame({'timeline': np.arange(0, 151, 1)})

#temp is a list of DataFrames (one per risk group), each with time as index and survival probability as value
#we want to left join each survival function onto the timeline by the index (time)
merged = timeline.copy()
for i, surv_func in enumerate(temp):
    # Reset index to get time as a column, rename survival prob column
    surv_df = surv_func.reset_index()
    surv_df.columns = ['timeline', f'surv_prob_{i+1}']
    merged = merged.merge(surv_df, on='timeline', how='left')

#merged now contains the timeline and survival probabilities for each group
print(merged.head(20))

In [ ]:
print(f"Best accuracy: {1-result.fun}")
print(f"Best parameters: {result.x}")

In [ ]:
#create risk labels based on patient surviving 119 months
def assign_risk_label(prob):
    """Assign EL1-EL10 based on survival probability"""
    if prob >= 0.9:
        return 'EL1'
    elif prob >= 0.8:
        return 'EL2'
    elif prob >= 0.7:
        return 'EL3'
    elif prob >= 0.6:
        return 'EL4'
    elif prob >= 0.5:
        return 'EL5'
    elif prob >= 0.4:
        return 'EL6'
    elif prob >= 0.3:
        return 'EL7'
    elif prob >= 0.2:
        return 'EL8'
    elif prob >= 0.1:
        return 'EL9'
    else:
        return 'EL10'

In [ ]:
#Produce KM plot of curve
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

def makeKMPlot(X_test_extended):
    #data used for KM plot
    km_data = pd.DataFrame({
        'time': X_test_extended['Survival_Time'],
        'event': X_test_extended['Event_Status'].astype(int), #0 - censored
        'risk_label': X_test_extended['Risk_Label']
    })

    #restrict plot to 10 years 
    max_time = 120
    km_data_10yr =km_data[km_data['time'] <= max_time]

    # Plot KM curves for each risk group
    plt.figure(figsize=(14,8))
    kmf = KaplanMeierFitter()

    #Define colours for each risk group
    color = ['#2ecc71', '#27ae60', '#f1c40f', '#f39c12', '#e67e22', 
            '#d35400', '#e74c3c', '#c0392b', '#8e44ad', '#2c3e50']

    risk_labels_sorted = [f'EL{i}' for i in range(1,11)]

    for i,label in enumerate(risk_labels_sorted):
        mask = km_data['risk_label'] == label
        n_patients = mask.sum()

        if n_patients > 0:
            kmf.fit(
                durations = km_data.loc[mask,'time'],
                event_observed = km_data.loc[mask,'event'],
                label = f'{label} (n={n_patients})'
                )
            kmf.plot_survival_function(
                color=color[i],
                linewidth=2.5,
                alpha=0.8
            )

In [ ]:
#Implement and Fit RSF
#free memory
#Takes time 4min
import gc
gc.collect()

rsf = RandomSurvivalForest(n_estimators=78, min_samples_leaf=5, max_features=20, max_samples=10000, n_jobs=13,verbose=1,random_state=42)
rsf.fit(X_train, y_train)



In [ ]:
#create KM Plots WITHOUT BATCHING
import numpy as np


#define time points (100 points between month 0 to 275)
time_points = np.linspace(0,275,100) 
#get integrated survival for each patient
integrated_survival_rsf = []

#get survival probability for each timepoint for each patient
survival_functions = rsf.predict_survival_function(X_test)
#calculate integrated survival for batch
for surv_func in survival_functions:
    surv_probs = surv_func(time_points)
    integrated_surv = surv_probs.mean()
    integrated_survival_rsf.append(integrated_surv)

#our integrated survival probabilities
integrated_survival = np.array(integrated_survival_rsf)

#generate labels

#Create DataFrame used for KM Plot
X_test_extended = pd.DataFrame(X_test,columns=X_test.columns)
print(f"X_test_extended shape: {X_test_extended.shape}")
risk_labels = [assign_risk_label(p) for p in integrated_survival]
X_test_extended['Risk_Label'] = risk_labels
X_test_extended['Survival_Time'] = y_test['time']
X_test_extended['Event_Status'] = y_test['event'].astype(np.float32)

km_plotter = KM_Plotter(X_test_extended)
km_plotter.makeKMPlot()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calculate_ece_rsf(rsf_model, X_test, durations_test, events_test, time_point=60, n_bins=10):
    """
    Calculate Expected Calibration Error (ECE) for RSF model at a specific time point
    """
    #get survival function predictions from RSF
    surv_funcs = rsf_model.predict_survival_function(X_test)
    
    #extract survival probability at time_point for each patient
    predicted_survival = []
    for fn in surv_funcs:
        # Find closest time point in the survival function
        times = fn.x
        probs = fn.y
        idx = np.abs(times - time_point).argmin()
        predicted_survival.append(probs[idx])
    
    predicted_survival = np.array(predicted_survival)
    
    #binary outcome: did patient survive past time_point?
    survived = ~((durations_test <= time_point) & (events_test == 1))
    
    #create equal-width bins
    bin_edges = np.linspace(0, 1, n_bins + 1)
    
    ece = 0.0
    bin_data = []
    
    for i in range(n_bins):
        mask = (predicted_survival >= bin_edges[i]) & (predicted_survival < bin_edges[i + 1])
        n_in_bin = mask.sum()
        
        if n_in_bin > 0:
            avg_predicted = predicted_survival[mask].mean()
            avg_observed = survived[mask].mean()
            bin_error = np.abs(avg_predicted - avg_observed)
            
            ece += bin_error * n_in_bin
            
            bin_data.append({
                'bin': i + 1,
                'lower': bin_edges[i],
                'upper': bin_edges[i + 1],
                'n_samples': n_in_bin,
                'avg_predicted': avg_predicted,
                'avg_observed': avg_observed,
                'error': bin_error
            })
    
    ece /= len(predicted_survival)
    
    return ece, pd.DataFrame(bin_data)


def plot_reliability_diagram_rsf(rsf_model, X_test, durations_test, events_test, time_point=60, n_bins=10):
    """
    Plot reliability diagram with ECE for RSF model
    """
    ece, bin_df = calculate_ece_rsf(rsf_model, X_test, durations_test, events_test, time_point, n_bins)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    #Reliability diagram
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.bar(bin_df['avg_predicted'], bin_df['avg_observed'], 
           width=0.08, alpha=0.7, label='Model calibration')
    ax.scatter(bin_df['avg_predicted'], bin_df['avg_observed'], 
               s=100, c='red', zorder=5)
    
    ax.set_xlabel('Mean Predicted Survival Probability', fontsize=12)
    ax.set_ylabel('Observed Survival Fraction', fontsize=12)
    ax.set_title(f'RSF Reliability Diagram at {time_point} Months\nECE = {ece:.4f}', 
                 fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'Report_Resources/rsf_ece_reliability_{time_point}m.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return ece, bin_df


#calculate ECE at different time points
print("=" * 60)
print("RSF - EXPECTED CALIBRATION ERROR (ECE)")
print("=" * 60)

#Assuming your RSF model is called 'rsf' and test data uses y_test structured array
durations_test_rsf = y_test['time']
events_test_rsf = y_test['event'].astype(float)

for t in [12, 36, 60, 120]:
    ece, bin_df = calculate_ece_rsf(rsf, X_test, durations_test_rsf, events_test_rsf, time_point=t)
    print(f"ECE at {t} months ({t//12} years): {ece:.4f}")

#plot reliability diagrams
print("\n" + "=" * 60)
print("RELIABILITY DIAGRAMS")
print("=" * 60)

ece_60, bin_df_60 = plot_reliability_diagram_rsf(rsf, X_test, durations_test_rsf, events_test_rsf, time_point=60)
print(f"\nBin details at 60 months:")
print(bin_df_60.to_string(index=False))

ece_120, bin_df_120 = plot_reliability_diagram_rsf(rsf, X_test, durations_test_rsf, events_test_rsf, time_point=120)
print(f"\nBin details at 120 months:")
print(bin_df_120.to_string(index=False))


In [ ]:
#get histogram of counts for each label
#get counts for each risk label
risk_counts = X_test_extended['Risk_Label'].value_counts()

#sort by label (EL1 to EL10)
risk_labels_sorted = [f'EL{i}' for i in range(1, 11)]
risk_counts_sorted = risk_counts.reindex(risk_labels_sorted)

#plot histogram
plt.figure(figsize=(12, 6))
plt.bar(risk_labels_sorted, risk_counts_sorted, color='steelblue', edgecolor='black')
plt.xlabel('Risk Label')
plt.ylabel('Number of Patients')
plt.title('Patient Distribution Across Risk Groups')

#add count labels on top of each bar
for i, count in enumerate(risk_counts_sorted):
    plt.text(i, count + 50, str(int(count)), ha='center', va='bottom')

plt.show()

#print counts
print(risk_counts_sorted)


In [ ]:
#get histogram for EL1 patients only to explain cutoff
X_test_EL1 = X_test_extended[X_test_extended["Risk_Label"] == "EL1"]

plt.figure(figsize=(12, 6))

#separate event and censored cases
el1_events = X_test_EL1[X_test_EL1['Event_Status'] == 1]['Survival_Time']
el1_censored = X_test_EL1[X_test_EL1['Event_Status'] == 0]['Survival_Time']

#plot overlapping histograms
plt.hist(el1_events, bins=30, alpha=0.7, label=f'Died (n={len(el1_events)})', color='red', edgecolor='black')
plt.hist(el1_censored, bins=30, alpha=0.7, label=f'Censored (n={len(el1_censored)})', color='green', edgecolor='black')

plt.xlabel('Survival Months')
plt.ylabel('Count')
plt.title('EL1 Patients: Survival Time Distribution by Event Status')
plt.legend()
plt.show()


In [ ]:
#Get weighting of each feature

#TODO: takes too long to run

from sklearn.inspection import permutation_importance
import numpy as np

#to avoid WinError spikes
rsf.set_params(n_jobs=1)

#get importance on test set
result = permutation_importance(
    rsf, X_test, y_test,
    n_repeats = 1,
    random_state=42,
    n_jobs=1,)

##Ask GPT for classification models, can use regression as well 
#Random forest can be processed categorically instead of binary
#Regression problem
#Looking at correlations between columns
#Identify columns with functional dependencies and rows
#width and length of column determines prediction power of column, so remove columns to improve power
#want many rows with less variables
#RSF not efficient enough since we can't get permutation importance
#It's a regression problem since output is continuous
#If I find a perfect existing model, then change problem slightly or extend it. 
#If we can't, then look at what's special in data and why models don't work. 
#Think about how I would talk about my approach in the presentation
#Could look into bespoke models - tweaking visual NNs

#Get more complex and more recent models (advanced models)

In [ ]:
#TODO: follows from code that takes too long to run

#could just take top 50 columns from X by importance and run this again
#display feature importance
# imp = pd.DataFrame({
#     "feature": X_test.columns,
#     "importance_mean": result.importances_mean,
#     "importance_std": result.importances_std,
# }).sort_values("importance_mean",ascending="False")

# print(imp.head(100))

In [ ]:
from collections import defaultdict
from scipy.sparse import issparse

#get feature names in the same order used by the input

feat_names = X_train.columns if not issparse(X_train) else np.arrange(X_train.shape[1]).astype(str)

n_features = len(feat_names)

#each index corresponds to column (feature)

#number of times feature (corresponding to index in list) appears in splits
split_counts = np.zeros(n_features,dtype=np.int64)

#every time we see a feature, we sum its depth 
depth_sums = np.zeros(n_features,dtype=np.int64)

#number of times we see feature
depth_hits = np.zeros(n_features,dtype=np.int64)

#mean depth = depth sums / depth hits

def accumulate_tree(tree):
    #.tree_.feature gives all the features belonging to each node 
    features = tree.tree_.feature
    #get left and right child nodes from root node of tree
    children_left = tree.tree_.children_left
    children_right = tree.tree_.children_right

    #count splits
    valid = features >= 0
    #get an array where each index is a corresponding feature and the value is the number of times the feature appears within our features array (up to the end, which is 'valid')
    counts = np.bincount(features[valid],minlength=n_features)
    return counts,children_left, children_right,features

def record_depths(node,depth,features,cl,cr):
    #feature corresponding to node 
    f = features[node]
    if f>=0: 
        depth_sums[f] += depth #add depth to depth sum of feature
        depth_hits[f] += 1 #increment hit

        #repeat for left child node and right child node
        record_depths(cl[node],depth+1,features,cl,cr)
        record_depths(cr[node],depth+1,features,cl,cr)

for est in model.estimators_:
    #cl - children left
    #counts - our bin array containing number of times each feature appears in split
    
    counts, cl, cr, features = accumulate_tree(est)
    split_counts += counts
    record_depths(0,0,features,cl,cr)

mean_depth = np.where(depth_hits > 0, depth_sums / depth_hits, np.inf)

imp_df = pd.DataFrame({
    "feature": feat_names,
    "split_count": split_counts,
    "mean_depth": mean_depth
}).sort_values(["split_count","mean_depth"], ascending=[False,True])

imp_df.head(20)

In [ ]:
print(imp_df.head(20).to_latex(index=False))


In [ ]:
#get pairwise correlation coefficient between top 25 features

import scipy.stats as ss

col_names = imp_df.head(25)["feature"].tolist()

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

feature_ones = []
feature_twos = []
cramers = []
for i in range(0,24):
    for j in range(0,24):
        if(j!=i):
            feature_ones.append(col_names[i])
            feature_twos.append(col_names[j])
            cramers.append(cramers_v(X[col_names[i]], X[col_names[j]]))
    


In [ ]:
#output correlations
correlations_df = pd.DataFrame({
    "feature_one": feature_ones,
    "feature_two": feature_twos,
    "cramer": cramers
}).sort_values('cramer', ascending=[False])

correlations_df

In [ ]:
pd.set_option("display.max_rows", None)
display(correlations_df)
pd.reset_option("display.max_rows")

#Look into semantics of columns to determine whether we should remove them 
#Look into why they gave high correlation

In [ ]:
#Get Pearsons correlation coefficient between age and survival 
#Using Pearsons correlation coefficient since both variables are numeric continuous
df2 = df

age_num  = pd.to_numeric(df2["Age recode with <1 year olds and 90+"], errors="coerce")
time_num = pd.to_numeric(df2["Survival months"], errors="coerce")

#Build a clean frame and drop rows with missing values in either column
d = pd.DataFrame({ "age": age_num, "surv_months": time_num })
d = d.dropna(subset=["age", "surv_months"])

#Pearson correlation coefficient
corr_pearson = d["age"].corr(d["surv_months"], method="pearson")
print(f"Pearson(age, survival months) = {corr_pearson:.3f}")

In [ ]:
#Scatter plot of Age vs Survival Months
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(d["age"], d["surv_months"], alpha=0.3, s=8)
plt.xlabel("Age (midpoint years)")
plt.ylabel("Survival months")
plt.title("Age vs Survival months")
plt.show()


In [ ]:
%pip install -U matplotlib

In [ ]:
#Age vs Predicted survival probability
# Choose a specific time point (in months)
time_point = 24

# Predict survival functions for all test patients
pred_surv = rsf.predict_survival_function(X_test)

# Extract survival probability at the chosen time
surv_prob_24m = np.array([fn(time_point) for fn in pred_surv])

In [ ]:
import seaborn as sns
# Plot Age vs predicted survival probability at 24 months
plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_test["Age recode with <1 year olds and 90+"],
    y=surv_prob_24m,
    alpha=0.4
)
plt.title(f"Predicted Survival Probability at {time_point} Months vs Age")
plt.xlabel("Age (midpoint years)")
plt.ylabel(f"Predicted Survival Probability at {time_point} Months")
plt.show()